In [1]:
#Librerías para la manipulación de los set de datos.
import os
import pandas as pd
import numpy as np
import random
#Librerías para el preprocesamiento y generación de la red neuronal LSTM.
import keras
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, Activation
from keras import optimizers, regularizers
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
#Librerías para la manipuloación de fechas
from datetime import datetime
import datetime as dts
from datetime import date
from datetime import timedelta, datetime
from os.path import exists


#Librería y función para suprimir los warning.
import sys
if not sys.warnoptions:
    import warnings
    warnings.simplefilter("ignore")
#Generación de semilla para que el modelo que se genere, con los mismos datos, sea el mismo, es decir, para garantizar replicación.
keras.utils.set_random_seed(4)

def gen_sequence(id_df, seq_length, seq_cols):
    df_zeros=pd.DataFrame(np.zeros((seq_length-1,id_df.shape[1])),columns=id_df.columns)
    id_df=df_zeros.append(id_df,ignore_index=True)
    data_array = id_df[seq_cols].values
    num_elements = data_array.shape[0]
    lstm_array=[]
    for start, stop in zip(range(0, num_elements-seq_length), range(seq_length, num_elements)):
        lstm_array.append(data_array[start:stop, :])
    return np.array(lstm_array)


path_variables = 'abfss://fsaavi40@dlaavi40.dfs.core.windows.net/'
file_name_variables ='Variables.xlsx'
variables = pd.read_excel(path_variables+file_name_variables)
dataframe = pd.read_csv('abfss://fsaavi40@dlaavi40.dfs.core.windows.net/Historical Data/HistoricalData.csv')
dataframe = dataframe.sort_values(by=['TIMESTAMP','Fecha_Generacion'],ascending=False).reset_index(drop=True)
dataframe = dataframe.iloc[dataframe[['TIMESTAMP']].drop_duplicates(keep='first').index]
dataframe = dataframe.sort_values(by=['TIMESTAMP'],ascending=True).reset_index(drop=True)
dataframe = dataframe.reset_index(drop=True)
dataframe = dataframe.drop(['Fecha_Generacion'], axis=1)
nuevos_datos = dataframe[(len(dataframe)-(90)):(len(dataframe)-10)]
fecha = str((datetime.today()-timedelta(hours=5)).year)+'-'+str((datetime.today()-timedelta(hours=5)).month)+'-'+str((datetime.today()-timedelta(hours=5)).day)
dataframe['TIMESTAMP'] = pd.to_datetime(dataframe['TIMESTAMP'])
dataframe = dataframe[(dataframe['TIMESTAMP']>(pd.to_datetime(fecha)-timedelta(days=7)))&(dataframe['TIMESTAMP']<(pd.to_datetime(fecha)))].reset_index(drop=True)
#dataframe = dataframe[0:10080].reset_index(drop=True)

pre = dataframe['PLC10_0_0'].astype(np.float)
post = dataframe['PLC10_0_1'].astype(np.float)+dataframe['PLC10_0_3'].astype(np.float)
post = post.replace(2,1)
data = pd.DataFrame()
data['TIMESTAMP'] = dataframe['TIMESTAMP']
data['Pre'] = pre
data['Post'] = post
data['Estado'] = ''

tiempo = 180
ventana_post_seco = 20
ventana_post = 60

for i in range(tiempo,len(data)):
    
    if data['Pre'][i]==1 :
        data['Estado'][i] = 'Reventón'

    if (data['Pre'][i]==0) & (sum(data['Pre'][(i-ventana_post):i])>0):
        data['Estado'][i] = 'Post'
        
    if (data['Pre'][i]==0) & (sum(data['Pre'][(i-ventana_post):i])==0) & (sum(data['Pre'][(i+1):(i+tiempo)])==0) :
    #if (data['Pre'][i]==0) & (sum(data['Pre'][(i-59):i])==0) & (sum(data['Post'][(i-59):i])==0) & (sum(data['Pre'][(i+1):(i+tiempo)])==0) :
        data['Estado'][i] = 'ESTABLE'
    
    if (data['Pre'][i]==0) & (sum(data['Pre'][(i-ventana_post):i])==0) & (sum(data['Pre'][(i+1):(i+tiempo)])>0) & (data['Estado'][i]=='') :
    #& (sum(data['Post'][(i-tiempo):i])==0)
        data['Estado'][i] = 'PRE'
        
    if (data['Post'][i]==1) & (sum(data['Pre'][i:(i+ventana_post_seco)])>0) & (data['Post'][i-1]==0) & (data['Pre'][i]==0) :
        data['Estado'][i:(i+15)] = 'Post'
        for j in range((i-(tiempo)),(i+ventana_post_seco)):
            if data['Estado'][j]=='PRE':
                data['Estado'][j] = 'ESTABLE'
    
    if data['Pre'][i]==1 :
        data['Estado'][i] = 'Reventón'

dataframe['Estado'] = data['Estado']

actual = (datetime.now()-timedelta(hours=5)).strftime("%H:%M")
if actual>'03:30':
    if actual<'03:55':
        Intervalos_confianza = pd.DataFrame()
        #Intervalos_confianza['Variables'] = (dataframe[dataframe['Estado']=='ESTABLE'].mean(axis=0)-3*dataframe[dataframe['Estado']=='ESTABLE'].std(axis=0)).index
        Intervalos_confianza['Variables']=list(dataframe.columns[1:-1])
        Intervalos_confianza['Máximo'] = np.percentile(dataframe[dataframe['Estado']=='ESTABLE'].iloc[:,1:(dataframe.shape[1]-1)], 75, axis=0)+1.5*(np.percentile(dataframe[dataframe['Estado']=='ESTABLE'].iloc[:,1:(dataframe.shape[1]-1)], 75, axis=0)-np.percentile(dataframe[dataframe['Estado']=='ESTABLE'].iloc[:,1:(dataframe.shape[1]-1)], 25, axis=0))
        Intervalos_confianza['Mínimo'] = np.percentile(dataframe[dataframe['Estado']=='ESTABLE'].iloc[:,1:(dataframe.shape[1]-1)], 25, axis=0)-1.5*(np.percentile(dataframe[dataframe['Estado']=='ESTABLE'].iloc[:,1:(dataframe.shape[1]-1)], 75, axis=0)-np.percentile(dataframe[dataframe['Estado']=='ESTABLE'].iloc[:,1:(dataframe.shape[1]-1)], 25, axis=0))
        Intervalos_confianza.to_csv('abfss://fsaavi40@dlaavi40.dfs.core.windows.net/Intervalos_confianza.csv',index=False)



entrenamiento = dataframe.reset_index(drop=True)
entrenamiento['reventon'] = 0
#indice =[5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,
#        35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,
#        65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94]

#indice = range(5,413)
dataset = entrenamiento
indice = np.array(variables[variables['Variables_usadas']==1]['TAG PHD'])
X = dataset[indice].astype(float)
#X = dataset.iloc[:,indice].astype(float)


X['id'] = 1



# marca de tiempo o tamaño de ventana 
seq_length=60
seq_cols=X.columns
sc=MinMaxScaler()
X[X.columns]=sc.fit_transform(X[X.columns])
X['id'] = 1


X_train=gen_sequence(X[X['id']==1], seq_length, seq_cols)
# Generamos el Y_train
entrenamiento = entrenamiento[(entrenamiento['Estado']=='ESTABLE') | (entrenamiento['Estado']=='PRE')]
entrenamiento.loc[entrenamiento['Estado']=='PRE','reventon'] = 1
X_train = X_train[entrenamiento.index[:(len(entrenamiento.index)-1)]]
Y = entrenamiento['reventon'].values
Y = np.asarray(Y).astype(np.float32)
y_train=Y


nb_features =X_train.shape[2]
timestamp=seq_length


model = Sequential()

model.add(LSTM(
            input_shape=(timestamp, nb_features),
            units=100,
            return_sequences=True))
model.add(LSTM(
            units=50,
            return_sequences=False))
model.add(Dense(units=1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
# ajustamos el modelo
model.fit(X_train, y_train, epochs=20, batch_size=200, validation_split=0.3, verbose=0,
            callbacks = [EarlyStopping(monitor='val_loss', min_delta=0, patience=0, verbose=0, mode='auto')])

In [2]:
n=20
n_variables_recomendacion=10
#Se ordenan los datos de forma ascendentes para la manipulación de estos.
nuevos_datos = nuevos_datos.sort_values(by=['TIMESTAMP'],ascending=True,)
nuevos_datos = nuevos_datos.reset_index(drop=True)
dataset_nuevos = nuevos_datos
X_nuevos = dataset_nuevos[indice].astype(float)
#X_nuevos = dataset_nuevos.iloc[:,indice].astype(float)

X_nuevos['id'] = 1
#Se transforman los datos con el modelo entrenado con el set de entrenamiento
X_nuevos[X_nuevos.columns]=sc.transform(X_nuevos[X_nuevos.columns])
X_nuevos['id'] = 1
#Se crean las secuencias de las variables explicativas con el histórico de 1 hora
#para cada una de ellas.
X_test=gen_sequence(X_nuevos[X_nuevos['id']==1], seq_length, seq_cols)
fecha = nuevos_datos['TIMESTAMP'][80-n:80]
predicciones = model.predict(X_test)[79-n:80]
resultados = pd.DataFrame()
resultados['Probabilidad'] = pd.DataFrame(predicciones).reset_index(drop=True)
resultados['Fecha'] = fecha.reset_index(drop=True)
data = nuevos_datos
data = data.reset_index(drop=True)
data = data.sort_values(by=['TIMESTAMP'],ascending=True,)
varianzas = data.copy()
coe_varia = data.copy()
ts_outliers = data.copy()
intervalos = data.copy()
##Se calcula cada una de las métricas en cada una de las variables en cada uno de los minutos considerados en este set de datos (predicciones)
Intervalos_confianza = pd.read_csv('abfss://fsaavi40@dlaavi40.dfs.core.windows.net/Intervalos_confianza.csv')
for i in range(60,len(data)):
    for j in range(4,data.shape[1]):
        mayores = sum(data.iloc[(i-60):i,j]>np.float(Intervalos_confianza[Intervalos_confianza['Variables']==data.columns[j]]['Máximo']))
        menores = sum(data.iloc[(i-60):i,j]<np.float(Intervalos_confianza[Intervalos_confianza['Variables']==data.columns[j]]['Mínimo']))
        intervalos.iloc[i,j] = mayores+menores 

        #varianza
        varianzas.iloc[i,j] = float( data.iloc[(i-60):i,j].std())
        #coeficiente
        coe_varia.iloc[i,j] = data.iloc[(i-60):i,j].std()/data.iloc[(i-60):i,j].mean()
        #Outliers
        temporal = (data.iloc[(i-60):i,j]-data.iloc[(i-60):i,j].mean())/data.iloc[(i-60):i,j].std()
        ts_outliers.iloc[i,j] = sum(temporal>3)

#Se consideran desde la fila 60 en adelanto debido a que como se mira el histórico de 60 minutos hacía atras, para estas observaciones no hay suficiente información.
varianzas = varianzas.iloc[60:(len(data)+1)]
coe_varia = coe_varia.iloc[60:(len(data)+1)]
ts_outliers = ts_outliers.iloc[60:(len(data)+1)]
intervalos = intervalos.iloc[60:(len(data)+1)]

#Se adjunta la fecha
sr_desviacion_estandar = pd.DataFrame()
sr_desviacion_estandar['Fecha'] = data['TIMESTAMP'].iloc[60:(len(data)+1)]
sr_coeficiente_variacion = pd.DataFrame()
sr_coeficiente_variacion['Fecha'] = data['TIMESTAMP'].iloc[60:(len(data)+1)]
sr_outliers = pd.DataFrame()
sr_outliers['Fecha'] = data['TIMESTAMP'].iloc[60:(len(data)+1)]
sr_intervalos = pd.DataFrame()
sr_intervalos['Fecha'] = data['TIMESTAMP'].iloc[60:(len(data)+1)]

numbers_names_variables = []
for i in range(n_variables_recomendacion):
    sr_desviacion_estandar['Variable '+str(i+1)] = ''
    sr_coeficiente_variacion['Variable '+str(i+1)] = ''
    sr_outliers['Variable '+str(i+1)] = ''
    sr_intervalos['Variable '+str(i+1)] = ''
    sr_desviacion_estandar['Variable '+str(i+1)+' Descripción'] = ''
    sr_coeficiente_variacion['Variable '+str(i+1)+' Descripción'] = ''
    sr_outliers['Variable '+str(i+1)+' Descripción'] = ''
    sr_intervalos['Variable '+str(i+1)+' Descripción'] = ''
    numbers_names_variables.append('Variable '+str(i+1))
    numbers_names_variables.append('Variable '+str(i+1)+' Descripción')

#Se ajusta el dataframe para obtener una estructura definida en cada uno de los tres dataframes para que no se confundan cuando se 
#condense todo en una sola tabla.
sr_desviacion_estandar = sr_desviacion_estandar.reset_index(drop=True)
sr_coeficiente_variacion = sr_coeficiente_variacion.reset_index(drop=True)
sr_outliers = sr_outliers.reset_index(drop=True)
sr_intervalos = sr_intervalos.reset_index(drop=True)
columnas = pd.DataFrame()
columnas['columnas']=data.columns

for i in range(len(columnas)):
    columnas['columnas'][i] = columnas['columnas'][i].replace(':','_')

variables = variables[variables['Variables_usadas']==1]
variables = variables.reset_index(drop=True)
indices = []
for i in range(len(variables)):
   if len(columnas[columnas['columnas']==variables['TAG'][i]])>0:
       indices.append(int(columnas[columnas['columnas']==variables['TAG'][i]].index[0]))

data.columns=columnas['columnas']
varianzas = varianzas.iloc[:,indices]
coe_varia = coe_varia.iloc[:,indices]
ts_outliers = ts_outliers.iloc[:,indices]
intervalos = intervalos.iloc[:,indices]


for i in range(len(varianzas)):
    orden_st = varianzas.iloc[i,4:varianzas.shape[1]].sort_values(ascending=False)
    orden_coef = coe_varia.iloc[i,4:coe_varia.shape[1]].sort_values(ascending=False)
    orden_outliers = ts_outliers.iloc[i,4:ts_outliers.shape[1]].sort_values(ascending=False)
    orden_intervalos = intervalos.iloc[i,4:intervalos.shape[1]].sort_values(ascending=False)
    
    for j in range(n_variables_recomendacion):
        sr_desviacion_estandar['Variable '+str(j+1)][i] = orden_st.index[j]
        sr_coeficiente_variacion['Variable '+str(j+1)][i] = orden_coef.index[j]
        sr_outliers['Variable '+str(j+1)][i] = orden_outliers.index[j]
        sr_intervalos['Variable '+str(j+1)][i] = orden_intervalos.index[j]

sr_outliers = sr_outliers.sort_values(by=['Fecha'],ascending=True)
sr_coeficiente_variacion = sr_coeficiente_variacion.sort_values(by=['Fecha'],ascending=True)
sr_desviacion_estandar = sr_desviacion_estandar.sort_values(by=['Fecha'],ascending=True)
sr_intervalos = sr_intervalos.sort_values(by=['Fecha'],ascending=True)

#Se concatena el nombre y el TAG para la visualización en POWER BI.
for i in range(len(sr_outliers)):
    for j in range(n_variables_recomendacion):
        sr_outliers[sr_outliers.columns[(j*2)+2]][i] = pd.DataFrame(variables[variables['TAG PHD']==sr_outliers[sr_outliers.columns[(j*2)+1]][i]]['Descripcion']).reset_index(drop=True)['Descripcion'][0]
        sr_coeficiente_variacion[sr_coeficiente_variacion.columns[(j*2)+2]][i] = pd.DataFrame(variables[variables['TAG PHD']==sr_coeficiente_variacion[sr_coeficiente_variacion.columns[(j*2)+1]][i]]['Descripcion']).reset_index(drop=True)['Descripcion'][0]
        sr_desviacion_estandar[sr_desviacion_estandar.columns[(j*2)+2]][i] = pd.DataFrame(variables[variables['TAG PHD']==sr_desviacion_estandar[sr_desviacion_estandar.columns[(j*2)+1]][i]]['Descripcion']).reset_index(drop=True)['Descripcion'][0]
        sr_intervalos[sr_intervalos.columns[(j*2)+2]][i] = pd.DataFrame(variables[variables['TAG PHD']==sr_intervalos[sr_intervalos.columns[(j*2)+1]][i]]['Descripcion']).reset_index(drop=True)['Descripcion'][0]

#Se eliminan los na
sr_outliers=sr_outliers.fillna('')
sr_coeficiente_variacion=sr_coeficiente_variacion.fillna('')
sr_desviacion_estandar=sr_desviacion_estandar.fillna('')
sr_intervalos=sr_intervalos.fillna('')

#Se concatena toda la información en un solo dataframe
for i in range(n_variables_recomendacion):
    resultados[sr_outliers.columns[2*i+1]+' Outliers']=sr_outliers[sr_outliers.columns[2*i+1]]+'-'+sr_outliers[sr_outliers.columns[2*i+2]]
    resultados[sr_desviacion_estandar.columns[2*i+1]+' DesvEstan']=sr_desviacion_estandar[sr_desviacion_estandar.columns[2*i+1]]+'-'+sr_desviacion_estandar[sr_desviacion_estandar.columns[2*i+2]]
    resultados[sr_coeficiente_variacion.columns[2*i+1]+' CoefVar']=sr_coeficiente_variacion[sr_coeficiente_variacion.columns[2*i+1]]+'-'+sr_coeficiente_variacion[sr_coeficiente_variacion.columns[2*i+2]]
    resultados[sr_intervalos.columns[2*i+1]+' IntCon']=sr_intervalos[sr_intervalos.columns[2*i+1]]+'-'+sr_intervalos[sr_intervalos.columns[2*i+2]]

resultados['Reventón']=0
resultados.loc[resultados['Probabilidad']>=0.5,'Reventón']=1
resultados['Fecha_actualizacion']=datetime.now()-timedelta(hours=5)
#Se exporta el archivo "exportable.csv" teniendo en cuenta la fecha y la fecha de la última acualización
path = 'abfss://fsaavi40@dlaavi40.dfs.core.windows.net/exportable.csv'

In [3]:
resultados_1 = pd.read_csv(path)
resultados = pd.concat([resultados,resultados_1])
resultados = resultados.sort_values(by=['Fecha','Fecha_actualizacion'],ascending=False).reset_index(drop=True)
resultados = resultados.iloc[resultados[['Fecha']].drop_duplicates(keep='first').index]
resultados = resultados.reset_index(drop=True)
path = 'abfss://fsaavi40@dlaavi40.dfs.core.windows.net/exportable.csv'
resultados[0:360].to_csv(path, index = False)
#Crear resultados_temporal y cambiar en el for resultados por resultados_temporal
#resultados_temporal = resultados[0:360]
#vector_exportable = pd.DataFrame()
#for i in range(0,10): 
#    vector_exportable = pd.concat([vector_exportable, resultados[resultados.columns[2+3*i]]]).reset_index(drop=True)

#vector_exportable.columns=['Variables']
#vector_exportable
#vector_exportable.to_csv('abfss://fsaavi40@dlaavi40.dfs.core.windows.net/Conteo_variables.csv', index=False)

In [4]:
path = 'abfss://fsaavi40@dlaavi40.dfs.core.windows.net/Minuto a minuto/Minuto a minuto'+(datetime.now()-timedelta(hours=5)).strftime("%Y-%m-%d")+' .csv'
actual = (datetime.now()-timedelta(hours=5)).strftime("%H:%M")
if actual>'00:00':
    if actual<'01:00':
        Seguimiento_1 = resultados
        Seguimiento_1 = Seguimiento_1.sort_values(by=['Fecha','Fecha_actualizacion'],ascending=False).reset_index(drop=True)
        Seguimiento_1 = Seguimiento_1.iloc[Seguimiento_1[['Fecha']].drop_duplicates(keep='first').index]
        Seguimiento_1 = Seguimiento_1.reset_index(drop=True)
        Seguimiento_1.to_csv(path)

Seguimiento_1 = pd.read_csv(path)
Seguimiento_1 = pd.concat([resultados,Seguimiento_1])
Seguimiento_1 = Seguimiento_1.sort_values(by=['Fecha','Fecha_actualizacion'],ascending=False).reset_index(drop=True)
Seguimiento_1 = Seguimiento_1.iloc[Seguimiento_1[['Fecha']].drop_duplicates(keep='first').index]
Seguimiento_1 = Seguimiento_1.reset_index(drop=True)
Seguimiento_1.to_csv(path)